# Ilustrasi Data Akuisisi dan Fault

Notebook ini menjawab dua pertanyaan pembimbing:

1. **Akuisisi data** — apa saja 10 langkah yang dilakukan, dan apa yang terjadi
   pada *missing value* serta nilai di luar rentang 0–100 %?
2. **Grafik** — bagaimana bentuk sinyal asli dibanding setelah disisipi fault,
   untuk semua kelas gangguan?

Isinya sengaja dibuat ringan: hanya pembersihan data dan simulasi fault, tanpa
pelatihan model. Data yang dipakai sama dengan notebook penelitian lainnya
(`data_sensor.csv`, 281.721 baris, 14 Sep–21 Des 2025, interval 30 detik).


## 1 — Memuat dan memeriksa data mentah

Sepuluh langkah di bawah ini adalah urutan yang sama dengan pipeline penelitian.
Kolom **Tindakan** menunjukkan mana yang membuang baris dan mana yang hanya
melaporkan.


In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

RANDOM_SEED = 42
SAMPLING_SECONDS = 300          # setelah rata-rata per 5 menit
RESAMPLE_RULE = "5min"
cols = ["kelembaban1", "kelembaban2", "kelembaban3", "kelembaban4"]
SENSORS = ["S1", "S2", "S3", "S4"]
os.makedirs("exports", exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/data_sensor.csv"
DATA_NAME = "data_sensor.csv"

def load_sensor_data():
    cands = [DATA_NAME, f"../input/{DATA_NAME}"] + glob.glob(f"/kaggle/input/**/{DATA_NAME}", recursive=True)
    for p in cands:
        if os.path.exists(p):
            print("Sumber data: file lokal ->", p)
            return pd.read_csv(p, index_col=0)
    print("Sumber data: unduh ->", DATA_URL)
    import requests
    r = requests.get(DATA_URL, timeout=180); r.raise_for_status()
    return pd.read_csv(StringIO(r.text), index_col=0)

def export_df(df, name):
    path = f"exports/{name}.csv"
    df.to_csv(path, index=False)
    print(path)

df_raw = load_sensor_data()
df_raw.index = pd.to_datetime(df_raw.index, utc=True)
df_raw = df_raw.sort_index()
print("Baris akuisisi:", len(df_raw))
print("Rentang waktu :", df_raw.index.min(), "->", df_raw.index.max())


In [ ]:
# --- Audit 10 langkah (angka dihitung dari data, bukan diketik manual) ---
audit = {}
audit["kolom_ada"] = sum(c in df_raw.columns for c in cols)
audit["baris_mentah"] = len(df_raw)

dup = int(df_raw.index.duplicated().sum())
audit["stempel_ganda"] = dup
df = df_raw[~df_raw.index.duplicated()].copy()

audit["nan_mentah"] = {c: int(df[c].isna().sum()) for c in cols}
audit["luar_range"] = {c: int(((df[c] < 0) | (df[c] > 100)).sum()) for c in cols}

def hitung_macet(s, minimal=10):
    s = pd.Series(s).reset_index(drop=True)
    sama = s.eq(s.shift())
    grup = (sama != sama.shift()).cumsum()
    panjang = sama.groupby(grup).sum()
    return int((panjang >= minimal - 1).sum())

audit["macet"] = {c: hitung_macet(df[c]) for c in cols}
audit["pencilan_z6"] = {c: int((np.abs((df[c] - df[c].mean()) / df[c].std()) > 6).sum()) for c in cols}

# Rata-rata per 5 menit -> grid seragam -> interpolasi waktu bila ada slot bolong
r = df[cols].resample(RESAMPLE_RULE).mean()
grid = pd.date_range(r.index[0], r.index[-1], freq=f"{SAMPLING_SECONDS}s")
r = r.reindex(grid)
audit["slot_bolong"] = int(r[cols].isna().all(axis=1).sum())
n_before = int(r[cols].isna().sum().sum())
df_sync = r.interpolate(method="time", limit_direction="both")
audit["isian_interpolasi"] = int(n_before - df_sync[cols].isna().sum().sum())
n_before2 = int(df_sync[cols].isna().sum().sum())
df_sync = df_sync.fillna(df_sync[cols].median())
audit["isian_median"] = int(n_before2 - df_sync[cols].isna().sum().sum())
audit["baris_akhir"] = len(df_sync)

print("Baris mentah :", audit["baris_mentah"])
print("Stempel ganda:", audit["stempel_ganda"], "-> dibuang")
print("NaN mentah   :", audit["nan_mentah"])
print("Luar 0-100%  :", audit["luar_range"], "-> dilaporkan, TIDAK dibuang")
print("Macet >=10   :", audit["macet"], "-> dilaporkan, TIDAK dibuang")
print("Pencilan |z|>6:", audit["pencilan_z6"], "-> dilaporkan, TIDAK dibuang")
print("Setelah 5 menit:", audit["baris_akhir"], "baris | slot bolong:", audit["slot_bolong"],
      "| isian interpolasi:", audit["isian_interpolasi"], "| isian median:", audit["isian_median"])


## 2 — Nasib tiap langkah di dataset ini

Pertanyaan pembimbing: *"missing value atau data di luar range diapain dulu di
tahap ini?"* — jawabannya ada di kolom **Tindakan**. Tidak ada baris yang
dibuang karena nilainya mencurigakan; yang dibuang hanya cacat struktural
(stempel waktu ganda), dan di dataset ini jumlahnya nol.

| # | Langkah | Hasil di data ini | Tindakan |
|---|---|---|---|
| 1 | Kolom wajib `kelembaban1..4` | 4/4 ada | lanjut, kalau tidak ada program berhenti |
| 2 | `datetime` UTC lalu diurutkan | 281.721 baris terurut | — |
| 3 | Stempel waktu ganda dibuang | **0** | tidak ada baris yang dibuang |
| 4 | Cek nilai kosong (NaN) | **0** per kolom | tidak ada yang perlu ditambal |
| 5 | Cek nilai di luar 0–100 % | **0** | **dilaporkan, tidak dibuang** — bisa jadi gejala fault |
| 6 | Pembacaan macet ≥ 10 berturut | S1 556 · S2 654 · S3 659 · S4 657 kejadian | **dilaporkan, tidak dibuang** |
| 7 | Pencilan ekstrem \|z\| > 6 | S1 13 · S2 0 · S3 0 · S4 0 titik | **dilaporkan, tidak dibuang** |
| 8 | Rata-rata per 5 menit | 281.721 → **28.173** baris | agregasi, bukan pembuangan |
| 9 | Grid seragam + interpolasi waktu | 0 slot bolong → **0** isian | tidak terpakai di dataset ini |
| 10 | Sisa NaN ditambal median | 0 → **0** isian | tidak terpakai di dataset ini |

**Jadi secara eksplisit:** missing value, *kalau ada*, ditambal di langkah 9
(interpolasi waktu) lalu langkah 10 (median kolom). Nilai di luar 0–100 % tidak
dibuang, hanya dicatat, karena pada penelitian ini pembacaan mustahil adalah
bukti fault, bukan sampah.

**Setelah injeksi fault** muncul NaN baru dari *hardware malfunction*
(`loss_prob = 5 %`). NaN itu bukan kotoran data akuisisi: di §3 nilainya
diisi `ffill/bfill` lalu median supaya fitur bisa dihitung, sementara `mask`
fault tetap disimpan agar label per-sensor tidak berbohong.


In [ ]:
# --- Tabel langkah 1-10 dalam bentuk tabel ---
tabel = pd.DataFrame([
    [1, "Kolom wajib kelembaban1..4", f"{audit['kolom_ada']}/4 ada", "lanjut"],
    [2, "datetime UTC lalu diurutkan", f"{audit['baris_mentah']:,} baris terurut", "-"],
    [3, "Stempel waktu ganda dibuang", f"{audit['stempel_ganda']}", "dibuang (0 baris)"],
    [4, "Cek nilai kosong (NaN)", str(sum(audit['nan_mentah'].values())), "tidak ada yang ditambal"],
    [5, "Cek nilai di luar 0-100%", str(sum(audit['luar_range'].values())), "dilaporkan, tidak dibuang"],
    [6, "Pembacaan macet >= 10", " / ".join(f"{k}:{v}" for k, v in audit['macet'].items()), "dilaporkan, tidak dibuang"],
    [7, "Pencilan ekstrem |z| > 6", " / ".join(f"{k}:{v}" for k, v in audit['pencilan_z6'].items()), "dilaporkan, tidak dibuang"],
    [8, "Rata-rata per 5 menit", f"{audit['baris_mentah']:,} -> {audit['baris_akhir']:,} baris", "agregasi"],
    [9, "Grid seragam + interpolasi waktu", f"slot bolong {audit['slot_bolong']} -> isian {audit['isian_interpolasi']}", "tidak terpakai di data ini"],
    [10, "Sisa NaN ditambal median", f"isian {audit['isian_median']}", "tidak terpakai di data ini"],
], columns=["#", "Langkah", "Hasil", "Tindakan"])
display(tabel)
export_df(tabel, "ILU_tabel_10_langkah")


In [ ]:
# --- Empat kanal setelah pembersihan (sebelum injeksi fault) ---
fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for ax, c, s in zip(axes, cols, SENSORS):
    ax.plot(df_sync.index, df_sync[c], lw=0.6, color="#34495e")
    ax.set_ylabel(s + " (%)", fontsize=8)
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("Waktu (UTC)")
fig.suptitle("Data setelah pembersihan dan penyelarasan waktu (5 menit)", fontsize=10)
fig.tight_layout()
fig.savefig("exports/ILU_empat_kanal_bersih.png", dpi=120, bbox_inches="tight")
plt.show()
print("Simpan: exports/ILU_empat_kanal_bersih.png")


In [ ]:
# === Simulator fault — parameter dan perilaku identik dengan notebook penelitian ===
DRIFT_PER_SAMPLE = 0.02 * (SAMPLING_SECONDS / 30)

def simulate_drift_fault(x, intensity=DRIFT_PER_SAMPLE, seed=None):
    drift = np.arange(len(x)) * intensity
    return x + drift, np.abs(drift) > 1e-6

def simulate_spike_fault(x, intensity=0.08, p=0.015, seed=None):
    tau = max(1, int(1.0 / p)) if p > 0 else len(x)
    spikes = (np.arange(len(x)) % tau == 0).astype(float) * (intensity * np.nanstd(x))
    return x + spikes, spikes != 0

def simulate_bias_fault(x, bias=0.08, seed=None):
    return x + bias, np.ones(len(x), bool)

def simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed=None):
    rng = np.random.default_rng(seed)
    n = len(x); rv = rng.random(n); idx = rng.integers(n, size=n)
    m1 = rv < stuck_prob; y = x.copy(); y[m1] = x[idx[m1]]
    m2 = rv < loss_prob; y[m2] = np.nan
    return y, (m1 | m2)

def simulate_multiple_faults(x, faults, seed=None):
    y = x.copy(); m_ = np.zeros(len(x), bool)
    for f, kw in faults:
        y, mi = f(y, **kw, seed=seed); m_ |= mi
    return y, m_

_D = {"intensity": DRIFT_PER_SAMPLE}
_S = {"intensity": 0.08, "p": 0.015}
_B = {"bias": 0.08}
_H = {"stuck_prob": 0.08, "loss_prob": 0.05}

SCENARIOS = {
    "normal": [],
    "drift": [(simulate_drift_fault, _D)],
    "spike": [(simulate_spike_fault, _S)],
    "bias": [(simulate_bias_fault, _B)],
    "hardware": [(simulate_hardware_fault, _H)],
    "bias+malfunc": [(simulate_bias_fault, _B), (simulate_hardware_fault, _H)],
    "spike+malfunc": [(simulate_spike_fault, _S), (simulate_hardware_fault, _H)],
    "spike+bias": [(simulate_spike_fault, _S), (simulate_bias_fault, _B)],
    "drift+malfunc": [(simulate_drift_fault, _D), (simulate_hardware_fault, _H)],
    "drift+bias": [(simulate_drift_fault, _D), (simulate_bias_fault, _B)],
    "drift+spike": [(simulate_drift_fault, _D), (simulate_spike_fault, _S)],
    "spike+bias+malfunc": [(simulate_spike_fault, _S), (simulate_bias_fault, _B), (simulate_hardware_fault, _H)],
    "drift+bias+malfunc": [(simulate_drift_fault, _D), (simulate_bias_fault, _B), (simulate_hardware_fault, _H)],
    "spike+drift+malfunc": [(simulate_spike_fault, _S), (simulate_drift_fault, _D), (simulate_hardware_fault, _H)],
    "drift+spike+bias": [(simulate_drift_fault, _D), (simulate_spike_fault, _S), (simulate_bias_fault, _B)],
    "spike+bias+malfunc+drift": [(simulate_spike_fault, _S), (simulate_bias_fault, _B),
                                  (simulate_hardware_fault, _H), (simulate_drift_fault, _D)],
}

print("Kondisi:", len(SCENARIOS), "| drift per sampel:", round(DRIFT_PER_SAMPLE, 4), "%")


## 3 — Grafik sinyal asli vs setelah injeksi fault

Permintaan pembimbing: tampilkan **grafik data asli dan data setelah disisipi
fault untuk semua kelas**, cukup sampel. Dua bentuk:

1. **Lima panel vertikal** per sensor — meniru contoh pembimbing: Spike, Drift,
   drift + spike, drift + spike + bias, dan keempat gangguan sekaligus.
2. **Grid 16 panel** per sensor — Normal + 15 kombinasi fault, jadi seluruh
   kelas tercakup.

Garis **abu-abu** = sinyal asli (setelah pembersihan, sebelum injeksi).
Garis **merah** = setelah injeksi; pada *hardware malfunction* NaN dibiarkan
sebagai **celah**. Garis **oranye putus-putus** = setelah NaN ditambal
`ffill/bfill` + median, yaitu yang benar-benar dilihat model. Potongan yang
diplot 300 sampel = 1,04 hari pada laju 5 menit; seed tetap supaya gambar
reproducible. Nilai yang diplot juga diekspor ke CSV.

**Catatan skala.** Drift menambah 0,2 % per sampel (≈ +60 % dalam 300 sampel),
jadi paling kasatmata. Sebaliknya spike hanya ±0,08 × simpangan baku dan bias
hanya +0,08 %, lebih kecil daripada ayunan alami sinyal; karena itu panel
spike/bias diberi kotak **selisih vs asli** (gangguan − asli) supaya besar
gangguan tetap terbaca. *Hardware malfunction* memakai 8 % sampel macet dan
5 % sampel hilang (NaN); rentang NaN diberi bayangan oranye.

Grafik ini hanya ilustrasi; ia tidak mengubah data yang dipakai §4.


In [ ]:
# === Grafik: sinyal asli vs setelah injeksi fault (contoh untuk laporan) ===
FIG_START = 2000          # potongan 300 sampel = 1,04 hari pada laju 5 menit
FIG_LEN = 300
CASES_5 = [
    ("Spike", "spike"),
    ("Drift", "drift"),
    ("Kombinasi dua gangguan: drift + spike", "drift+spike"),
    ("Kombinasi tiga gangguan: drift + spike + bias", "drift+spike+bias"),
    ("Kombinasi empat gangguan: drift + spike + bias + hardware malfunction",
     "spike+bias+malfunc+drift"),
]
CASES_GRID = [
    ("Normal (tanpa gangguan)", None),
    ("spike", "spike"), ("drift", "drift"), ("bias", "bias"), ("hardware", "hardware"),
    ("drift + spike", "drift+spike"), ("drift + bias", "drift+bias"),
    ("drift + hardware", "drift+malfunc"), ("spike + bias", "spike+bias"),
    ("spike + hardware", "spike+malfunc"), ("bias + hardware", "bias+malfunc"),
    ("drift + spike + bias", "drift+spike+bias"),
    ("drift + spike + hardware", "spike+drift+malfunc"),
    ("drift + bias + hardware", "drift+bias+malfunc"),
    ("spike + bias + hardware", "spike+bias+malfunc"),
    ("drift + spike + bias + hardware", "spike+bias+malfunc+drift"),
]

def _inject_plot(x, key, seed):
    if key is None:
        return x.copy(), x.copy()
    y, _ = simulate_multiple_faults(x, SCENARIOS[key], seed=seed)
    yf = pd.DataFrame(y).ffill().bfill()
    yf = yf.fillna(yf.median(numeric_only=True)).to_numpy().ravel()
    return np.asarray(y, dtype=float), np.asarray(yf, dtype=float)

def _panel(ax, x, y_asli, y_inj, y_isi, judul, inset=False):
    ax.plot(x, y_asli, color="#8a8a8a", lw=2.0, alpha=0.75, label="Sinyal asli")
    ax.plot(x, y_inj, color="#c0392b", lw=1.0, label="Setelah disisipi gangguan")
    if np.isnan(y_inj).any():
        ax.plot(x, y_isi, color="#e67e22", lw=1.0, ls="--",
                label="Setelah NaN ditambal (ffill + median)")
        nan_idx = np.where(np.isnan(y_inj))[0]
        runs = np.split(nan_idx, np.where(np.diff(nan_idx) != 1)[0] + 1)
        for j, run in enumerate(runs):
            if len(run) == 0:
                continue
            ax.axvspan(run[0] - 0.5, run[-1] + 0.5, color="#e67e22", alpha=0.18,
                       lw=0, label="Rentang NaN (data hilang)" if j == 0 else None)
    dif = y_inj - y_asli
    if inset and not np.isnan(dif).any() and np.nanmax(np.abs(dif)) < 1.0 and np.nanmax(np.abs(dif)) > 0:
        ki = int(np.nanargmax(np.abs(dif)))
        lo, hi = max(0, ki - 25), min(len(x), ki + 26)
        axi = ax.inset_axes([0.60, 0.52, 0.37, 0.43])
        axi.plot(x[lo:hi], dif[lo:hi], color="#c0392b", lw=1.2)
        axi.axhline(0, color="#8a8a8a", lw=0.8, ls=":")
        axi.tick_params(labelsize=5)
        axi.set_title("selisih vs asli (maks %.3g%%)" % np.nanmax(np.abs(dif)),
                      fontsize=6)
        axi.grid(alpha=0.2)
    ax.set_title(judul, fontsize=9)
    ax.margins(y=0.08)
    ax.grid(alpha=0.25)

def _catat(rows, sensor, kasus, x, asli, inj, isi):
    for xi, a, b, d in zip(x, asli, inj, isi):
        rows.append({"Sensor": sensor, "Kasus": kasus, "Indeks": int(xi),
                     "Asli": round(float(a), 4),
                     "Injeksi": (np.nan if np.isnan(b) else round(float(b), 4)),
                     "Terisi": round(float(d), 4)})

x_plot = np.arange(FIG_LEN)
plot_rows = []
for si, (c, sname) in enumerate(zip(cols, SENSORS)):
    seg = df_sync[c].iloc[FIG_START:FIG_START + FIG_LEN].to_numpy(dtype=float)
    if len(seg) < FIG_LEN:
        raise ValueError("Potongan data untuk gambar kurang dari 300 sampel.")

    # --- (1) lima panel vertikal, gaya contoh pembimbing ---
    fig, axes = plt.subplots(5, 1, figsize=(7.2, 12.0), sharex=True)
    for ci, (ax, (judul, key)) in enumerate(zip(axes, CASES_5)):
        y_inj, y_isi = _inject_plot(seg, key, seed=9000 + si * 100 + ci)
        _panel(ax, x_plot, seg, y_inj, y_isi, judul, inset=True)
        ax.set_ylabel("Kelembaban (%)", fontsize=8)
        _catat(plot_rows, sname, key, x_plot, seg, y_inj, y_isi)
    axes[-1].set_xlabel("Indeks sampel (5 menit)")
    h, l = [], []
    for _ax in axes:
        for _h, _l in zip(*_ax.get_legend_handles_labels()):
            if _l not in l:
                h.append(_h); l.append(_l)
    fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.965),
               ncol=4, fontsize=7.5, frameon=False)
    fig.suptitle("Contoh sinyal asli vs setelah injeksi fault — " + sname,
                 y=0.995, fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig("exports/ILU_contoh_fault_vs_asli_" + sname + ".png",
                dpi=120, bbox_inches="tight")
    plt.show()
    print("[Tersimpan] exports/ILU_contoh_fault_vs_asli_" + sname + ".png")

    # --- (2) grid 16 panel: normal + 15 kombinasi ---
    fig, axes = plt.subplots(4, 4, figsize=(14, 11), sharex=True)
    for ci, (ax, (judul, key)) in enumerate(zip(axes.ravel(), CASES_GRID)):
        y_inj, y_isi = _inject_plot(seg, key, seed=9000 + 5000 + si * 100 + ci)
        _panel(ax, x_plot, seg, y_inj, y_isi, judul)
        ax.tick_params(labelsize=7)
        _catat(plot_rows, sname, key or "normal", x_plot, seg, y_inj, y_isi)
    for ax in axes[-1]:
        ax.set_xlabel("Indeks sampel (5 menit)", fontsize=8)
    h, l = [], []
    for _ax in axes.ravel():
        for _h, _l in zip(*_ax.get_legend_handles_labels()):
            if _l not in l:
                h.append(_h); l.append(_l)
    fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.965),
               ncol=4, fontsize=7.5, frameon=False)
    fig.suptitle("Semua kelas: sinyal asli vs setelah injeksi fault — " + sname,
                 y=0.998, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig("exports/ILU_grid_fault_vs_asli_" + sname + ".png",
                dpi=120, bbox_inches="tight")
    plt.show()
    print("[Tersimpan] exports/ILU_grid_fault_vs_asli_" + sname + ".png")

plot_df = pd.DataFrame(plot_rows)
export_df(plot_df, "ILU_contoh_fault_vs_asli")
print("Baris data plot:", len(plot_df), "| sensor:", sorted(plot_df.Sensor.unique()))


## 4 — Ringkasan

- Dari 281.721 baris mentah tidak ada baris yang dibuang karena nilainya
  mencurigakan: stempel ganda 0, NaN 0, nilai di luar 0–100 % 0. Nilai yang
  mencurigakan (pembacaan macet, pencilan ekstrem) **dilaporkan** karena pada
  penelitian ini justru merupakan gejala fault.
- Setelah rata-rata 5 menit, data menjadi 28.173 baris pada grid seragam;
  tidak ada slot bolong sehingga interpolasi dan isian median tidak terpakai.
- Grafik di §3 memakai parameter injeksi yang sama dengan notebook penelitian.
  Perhatikan skala: drift paling kasatmata, sedangkan spike dan bias sangat
  kecil terhadap ayunan sinyal, sehingga panelnya dilengkapi kotak selisih.
